#  1.0 Set Up Environment

In [ ]:
# Install PySpark
!pip install pyspark

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Directory structure on Drive:

In [ ]:
import os

base = '/content/drive/MyDrive/salary_pipeline'
for folder in ['data/bronze', 'data/silver', 'data/gold', 'data/output']:
    os.makedirs(f'{base}/{folder}', exist_ok=True)

print("Directories created.")

Directories created.


# 2.0 Download the Dataset from Kaggle into Colab:

In [ ]:
# Install Kaggle API
!pip install kaggle

# Upload your kaggle.json API token (get it from kaggle.com → Account → API)
from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d bambrozim/public-employees-salaries-from-brazilsp-2012
!unzip public-employees-salaries-from-brazilsp-2012.zip -d /content/raw_data/

Saving kaggle (1).json to kaggle (1) (2).json
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/bambrozim/public-employees-salaries-from-brazilsp-2012
License(s): CC0-1.0
public-employees-salaries-from-brazilsp-2012.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  public-employees-salaries-from-brazilsp-2012.zip
replace /content/raw_data/all_merged/2012_merged.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/raw_data/all_merged/2013_merged.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/raw_data/all_merged/2014_merged.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/raw_data/all_merged/2015_merged.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/raw_data/all_merged/2016_merged.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/raw_data/all_merged/201

# 3.0 Intialize SparkSession (Bronze Layer)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

spark = SparkSession.builder \
    .appName("BrazilSalaryPipeline") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# Define explicit schema (inferSchema=False)
salary_schema = StructType([
    StructField("NAME", StringType(), True),
    StructField("POSITION", StringType(), True),
    StructField("DEPARTMENT", StringType(), True),
    StructField("MONTH_TOTAL", StringType(), True),
    StructField("GROSS_TOTAL", StringType(), True),
    StructField("NET_TOTAL", StringType(), True),
    StructField("MONTH", StringType(), True),
    StructField("YEAR", IntegerType(), True)
])

# Load each year — adjust column names after inspecting actual CSV headers
df_2013 = spark.read.option("header", True)\
    .option("sep", ";")\
    .schema(salary_schema)\
    .csv('/content/raw_data/all_merged/2013_merged.csv')

df_2014 = spark.read.option("header", True)\
    .option("sep", ";")\
    .schema(salary_schema)\
    .csv('/content/raw_data/all_merged/2014_merged.csv')

df_2015 = spark.read.option("header", True)\
    .option("sep", ";")\
    .schema(salary_schema)\
    .csv('/content/raw_data/all_merged/2015_merged.csv')

# Write to Bronze (raw, immutable)
base = '/content/drive/MyDrive/salary_pipeline'
df_2013.write.mode('overwrite').parquet(f'{base}/data/bronze/2013')
df_2014.write.mode('overwrite').parquet(f'{base}/data/bronze/2014')
df_2015.write.mode('overwrite').parquet(f'{base}/data/bronze/2015')

print("Bronze layer written.")

df_2013.show(5, truncate=False)
df_2013.printSchema()

Bronze layer written.
+--------------------------+------------------------------+----------------------------------------+-----------+-----------+---------+-----+----+
|NAME                      |POSITION                      |DEPARTMENT                              |MONTH_TOTAL|GROSS_TOTAL|NET_TOTAL|MONTH|YEAR|
+--------------------------+------------------------------+----------------------------------------+-----------+-----------+---------+-----+----+
|AAIRON TELES DE CAMARGO   |OFICIAL ADMINISTRATIVO        |PLANEJAMENTO E DESENVOLVIMENTO REGIONAL |1220,42    |2323,37    |1711,68  |APRIL|2013|
|AAIRON TELES DE CAMARGO   |OFICIAL ADMINISTRATIVO        |DEPARTAMENTO ESTADUAL DE TRANSITO-DETRAN|600,00     |600,00     |522,00   |APRIL|2013|
|AARAN ESTEVAO LIMA BARBOSA|2TEN  PM                      |POLICIA MILITAR ESTADO SAO PAULO        |5996,86    |5996,86    |4453,91  |APRIL|2013|
|AARAO DE OLIVEIRA         |MAJ   PM                      |POLICIA MILITAR ESTADO SAO PAULO        |11

In [ ]:
# See if the DataFrame itself has nulls where they shouldn't be (sign of misaligned parsing)
df_2013.select("NAME", "POSITION", "DEPARTMENT").summary("count").show()

# Sample rows from later in the file, not just the top
df_2013.sample(0.0001).show(20, truncate=False)

# Directly find rows where NAME looks abnormally long (a sign multiple fields got merged in)
from pyspark.sql.functions import col, length
df_2013.filter(length(col("NAME")) > 50).show(10, truncate=False)

+-------+--------+--------+----------+
|summary|    NAME|POSITION|DEPARTMENT|
+-------+--------+--------+----------+
|  count|11617364|11617364|  11617364|
+-------+--------+--------+----------+

+------------------------------+---------------------------------------+--------------------------------------+-----------+-----------+---------+-----+----+
|NAME                          |POSITION                               |DEPARTMENT                            |MONTH_TOTAL|GROSS_TOTAL|NET_TOTAL|MONTH|YEAR|
+------------------------------+---------------------------------------+--------------------------------------+-----------+-----------+---------+-----+----+
|AIRTON FERNANDES              |PROFESSOR EDUCACAO BASICA II           |EDUCACAO                              |5359,59    |5359,59    |3842,36  |APRIL|2013|
|ALESSANDRO ROSINI             |PROFESSOR EDUCACAO BASICA II           |EDUCACAO                              |1228,47    |1842,70    |1593,26  |APRIL|2013|
|ALZIRA CANELLA DE 

In [ ]:
# df_2013.select("POSITION").distinct().count()
# df_2013.select("DEPARTMENT").distinct().count()

In [ ]:
# print("Distinct POSITION count:", df_2013.select("POSITION").distinct().count())

In [ ]:
# Combine all years' distinct POSITION and DEPARTMENT values before translating
# (ensures translation table covers 2013+2014+2015)

all_positions = df_2013.select("POSITION") \
    .union(df_2014.select("POSITION")) \
    .union(df_2015.select("POSITION")) \
    .distinct()

all_departments = df_2013.select("DEPARTMENT") \
    .union(df_2014.select("DEPARTMENT")) \
    .union(df_2015.select("DEPARTMENT")) \
    .distinct()

print("Total distinct POSITION across 2013-2015:", all_positions.count())
print("Total distinct DEPARTMENT across 2013-2015:", all_departments.count())

positions_pd = all_positions.toPandas()
departments_pd = all_departments.toPandas()

positions_pd.to_csv('/content/positions_distinct.csv', index=False)
departments_pd.to_csv('/content/departments_distinct.csv', index=False)

print("Saved distinct value lists for translation.")

Total distinct POSITION across 2013-2015: 4452
Total distinct DEPARTMENT across 2013-2015: 110
Saved distinct value lists for translation.


# 4.0 Silver Layer: Clean & Standardize

In [ ]:
departments_pd = departments_pd.sort_values("DEPARTMENT").reset_index(drop=True)
for i, row in departments_pd.iterrows():
    print(row["DEPARTMENT"])

ADMINISTRACAO GERAL DO ESTADO
ADMINISTRACAO PENITENCIARIA
AG.REGUL.SERV.PUBL.DEL.TRANSP.SP-ARTESP
AGENCIA METROPOL.BAIXADA SANTISTA-AGEM
AGENCIA METROPOLIT.BAIXADA SANTISTA-AGEM
AGENCIA METROPOLITANA CAMPINAS-AGEMCAMP
AGENCIA METROPOLITANA DA BAIXADA SANTIST
AGENCIA METROPOLITANA DE CAMPINAS - AGEM
AGENCIA METROPOLITANA VALE PARAIBA LITOR
AGENCIA REG.SANEA.ENERGIA E.SP-ARSESP
AGENCIA REG.SERV.P.DEL.TRANS.E.SP-ARTESP
AGENCIA REGUL.SANEAM. E ENERGIA-ARSESP
AGRICULTURA ABASTECIMENTO
CAIXA BENEFICENTE DA POLICIA MILITAR
CASA CIVIL
CENTRO EDUC. TECNOL.PAULA SOUZA-CEETEPS
CIA DE PROCESSAM.DADOS EST. SP-PRODESP
CIA DESENVOLVIMENTO AGRICOLA SP-CODASP
CIA PAULISTA OBRAS E SERVICOS-CPOS
CIA. AMBIENTAL EST. SP - CETESB
CIA. PROCES. DADOS EST.S.P.-PRODESP
CIA.DE TECNOL.SANEAMEN AMBIENTAL-CETESB
CIA.DESENV.HABITAC.URBANO EST. SP-CDHU
CIA.DO METROPOLITANO DE SAO PAULO-METRO
CIA.PAULISTA DE SECURITIZACAO-CPSEC
CIA.PAULISTA DE TRENS METROPOLITANO-CPTM
CIA.SANEAMENTO BASICO EST.S.PAULO-SABESP
CIA.SEGUR

In [ ]:
from pyspark.sql.functions import col, regexp_replace, sha2, concat_ws, trim, upper, count, when
from pyspark.sql.types import FloatType

# NEW: Department translation mapping (Portuguese -> English)
department_translation = spark.createDataFrame([
    ("ADMINISTRACAO GERAL DO ESTADO", "STATE GENERAL ADMINISTRATION"),
    ("ADMINISTRACAO PENITENCIARIA", "PENITENTIARY ADMINISTRATION"),
    ("AG.REGUL.SERV.PUBL.DEL.TRANSP.SP-ARTESP", "SP TRANSPORT REGULATORY AGENCY - ARTESP"),
    ("AGENCIA METROPOL.BAIXADA SANTISTA-AGEM", "BAIXADA SANTISTA METROPOLITAN AGENCY - AGEM"),
    ("AGENCIA METROPOLIT.BAIXADA SANTISTA-AGEM", "BAIXADA SANTISTA METROPOLITAN AGENCY - AGEM"),
    ("AGENCIA METROPOLITANA CAMPINAS-AGEMCAMP", "CAMPINAS METROPOLITAN AGENCY - AGEMCAMP"),
    ("AGENCIA METROPOLITANA DA BAIXADA SANTIST", "BAIXADA SANTISTA METROPOLITAN AGENCY - AGEM"),
    ("AGENCIA METROPOLITANA DE CAMPINAS - AGEM", "CAMPINAS METROPOLITAN AGENCY - AGEMCAMP"),
    ("AGENCIA METROPOLITANA VALE PARAIBA LITOR", "VALE DO PARAIBA AND COAST METROPOLITAN AGENCY"),
    ("AGENCIA REG.SANEA.ENERGIA E.SP-ARSESP", "SP SANITATION AND ENERGY REGULATORY AGENCY - ARSESP"),
    ("AGENCIA REG.SERV.P.DEL.TRANS.E.SP-ARTESP", "SP TRANSPORT REGULATORY AGENCY - ARTESP"),
    ("AGENCIA REGUL.SANEAM. E ENERGIA-ARSESP", "SP SANITATION AND ENERGY REGULATORY AGENCY - ARSESP"),
    ("AGRICULTURA ABASTECIMENTO", "AGRICULTURE AND SUPPLY"),
    ("CAIXA BENEFICENTE DA POLICIA MILITAR", "MILITARY POLICE BENEFIT FUND"),
    ("CASA CIVIL", "CIVIL HOUSE (GOVERNOR'S OFFICE)"),
    ("CENTRO EDUC. TECNOL.PAULA SOUZA-CEETEPS", "PAULA SOUZA TECHNICAL EDUCATION CENTER - CEETEPS"),
    ("CIA DE PROCESSAM.DADOS EST. SP-PRODESP", "SP DATA PROCESSING COMPANY - PRODESP"),
    ("CIA DESENVOLVIMENTO AGRICOLA SP-CODASP", "SP AGRICULTURAL DEVELOPMENT COMPANY - CODASP"),
    ("CIA PAULISTA OBRAS E SERVICOS-CPOS", "PAULISTA WORKS AND SERVICES COMPANY - CPOS"),
    ("CIA. AMBIENTAL EST. SP - CETESB", "SP ENVIRONMENTAL COMPANY - CETESB"),
    ("CIA. PROCES. DADOS EST.S.P.-PRODESP", "SP DATA PROCESSING COMPANY - PRODESP"),
    ("CIA.DE TECNOL.SANEAMEN AMBIENTAL-CETESB", "SP ENVIRONMENTAL SANITATION TECHNOLOGY COMPANY - CETESB"),
    ("CIA.DESENV.HABITAC.URBANO EST. SP-CDHU", "SP URBAN HOUSING DEVELOPMENT COMPANY - CDHU"),
    ("CIA.DO METROPOLITANO DE SAO PAULO-METRO", "SAO PAULO METRO COMPANY - METRO"),
    ("CIA.PAULISTA DE SECURITIZACAO-CPSEC", "PAULISTA SECURITIZATION COMPANY - CPSEC"),
    ("CIA.PAULISTA DE TRENS METROPOLITANO-CPTM", "PAULISTA METROPOLITAN TRAINS COMPANY - CPTM"),
    ("CIA.SANEAMENTO BASICO EST.S.PAULO-SABESP", "SP BASIC SANITATION COMPANY - SABESP"),
    ("CIA.SEGUROS DO EST. DE SAO PAULO-COSESP", "SAO PAULO STATE INSURANCE COMPANY - COSESP"),
    ("COMPANHIA DOCAS DE SAO SEBASTIAO", "SAO SEBASTIAO DOCKS COMPANY"),
    ("COMPANHIA ENERGETICA DE SAO PAULO-CESP", "SAO PAULO ENERGY COMPANY - CESP"),
    ("COMPANHIA PAUL. EVENTOS E TURISMO-CPETUR", "PAULISTA EVENTS AND TOURISM COMPANY - CPETUR"),
    ("COMPANHIA PAULISTA DE PARCEIRAS-CPP", "PAULISTA HOLDING COMPANY - CPP"),
    ("COMPANHIA PAULISTA DE PARCERIAS", "PAULISTA HOLDING COMPANY - CPP"),
    ("COMPANHIA PAULISTA DE SECURITIZACAO", "PAULISTA SECURITIZATION COMPANY - CPSEC"),
    ("CULTURA", "CULTURE"),
    ("DEPARTAMENTO AEROVIARIO EST. SP-DAESP", "SP STATE AIRWAY DEPARTMENT - DAESP"),
    ("DEPARTAMENTO ESTADUAL DE TRANSITO-DETRAN", "STATE TRAFFIC DEPARTMENT - DETRAN"),
    ("DEPARTAMENTO ESTRADA RODAGEM-DER", "STATE ROAD DEPARTMENT - DER"),
    ("DEPTO.DE AGUAS E ENERGIA ELETRICA-DAEE", "WATER AND ELECTRIC ENERGY DEPARTMENT - DAEE"),
    ("DESENV. ECON.,CIENC.,TECNOLOG E INOVACAO", "ECONOMIC DEVELOPMENT, SCIENCE, TECHNOLOGY AND INNOVATION"),
    ("DESENVOLVE SP -AGEN. DE DESENV. PAULISTA", "DESENVOLVE SP - PAULISTA DEVELOPMENT AGENCY"),
    ("DESENVOLVIMENTO ECON.,CIENCIA E TECNOL.", "ECONOMIC DEVELOPMENT, SCIENCE AND TECHNOLOGY"),
    ("DESENVOLVIMENTO METROPOLITANO", "METROPOLITAN DEVELOPMENT"),
    ("DESENVOLVIMENTO RODOVIARIO S/A-DERSA", "ROAD DEVELOPMENT COMPANY - DERSA"),
    ("DESENVOLVIMENTO SOCIAL", "SOCIAL DEVELOPMENT"),
    ("DIREITOS DA PESSOA COM DEFICIENCIA", "RIGHTS OF PERSONS WITH DISABILITIES"),
    ("EDUCACAO", "EDUCATION"),
    ("EMP.PAULISTA PLANEJ.METROPLITANO S.A-EMP", "PAULISTA METROPOLITAN PLANNING COMPANY - EMPLASA"),
    ("EMP.PTA. PLAN.METROPLITANO S.A-EMPLASA", "PAULISTA METROPOLITAN PLANNING COMPANY - EMPLASA"),
    ("EMPR METROPOL AGUAS E ENERGIA S/A-EMAE", "METROPOLITAN WATER AND ENERGY COMPANY - EMAE"),
    ("EMPREGO E RELACOES DO TRABALHO", "EMPLOYMENT AND LABOR RELATIONS"),
    ("EMPRESA METROP.TRANS.URB.S.PAULO-EMTU", "SAO PAULO METROPOLITAN URBAN TRANSPORT COMPANY - EMTU"),
    ("EMPRESA PLANEJ. METROPOLITANO-EMPLASA", "PAULISTA METROPOLITAN PLANNING COMPANY - EMPLASA"),
    ("ENERGIA", "ENERGY"),
    ("ESPORTE, LAZER E JUVENTUDE", "SPORTS, LEISURE AND YOUTH"),
    ("FACULDADE DE MEDICINA DE MARILIA-FAMEMA", "MARILIA MEDICAL SCHOOL - FAMEMA"),
    ("FACULDADE MEDICINA SAO JOSE DO RIO PRETO", "SAO JOSE DO RIO PRETO MEDICAL SCHOOL"),
    ("FAZENDA", "STATE TREASURY (FINANCE)"),
    ("FUND.AMPARO PESQUISA S.PAULO-FAPESP", "SAO PAULO RESEARCH SUPPORT FOUNDATION - FAPESP"),
    ("FUND.CONSERV.PROD.FLORESTAL S.PAULO", "SAO PAULO FOREST PRODUCTION CONSERVATION FOUNDATION"),
    ("FUND.CTO.AT.SOCIO-EDUC.ADOL.FUND.CASA-SP", "SP JUVENILE SOCIO-EDUCATIONAL CARE FOUNDATION - FUNDACAO CASA"),
    ("FUND.DESENV. ADMINISTRATIVO-FUNDAP", "ADMINISTRATIVE DEVELOPMENT FOUNDATION - FUNDAP"),
    ("FUND.DESENV. DA EDUCACAO-FDE", "EDUCATION DEVELOPMENT FOUNDATION - FDE"),
    ("FUND.INST.TERRAS JOSE G. DA SILVA-ITESP", "JOSE GOMES DA SILVA LAND INSTITUTE FOUNDATION - ITESP"),
    ("FUND.MEMORIAL DA AMERICA LATINA", "LATIN AMERICA MEMORIAL FOUNDATION"),
    ("FUND.ONCOCENTRO DE S.PAULO", "SAO PAULO ONCOLOGY CENTER FOUNDATION"),
    ("FUND.PADRE ANCHIETA-CP RAD TV EDUCATIVA", "PADRE ANCHIETA FOUNDATION - EDUCATIONAL RADIO/TV"),
    ("FUND.PARA O REMEDIO POPULAR-FURP", "POPULAR MEDICINE FOUNDATION - FURP"),
    ("FUND.PREV.COMPL.EST.SP-PREVCOM", "SP STATE SUPPLEMENTARY PENSION FOUNDATION - PREVCOM"),
    ("FUND.PRF.DR.MANOEL PEDRO PIMENTEL-FUNAP", "DR. MANOEL PEDRO PIMENTEL PROFESSOR FOUNDATION - FUNAP"),
    ("FUND.PRO-SANGUE-HEMOCENTRO S.PAULO", "PRO-BLOOD FOUNDATION - SAO PAULO BLOOD CENTER"),
    ("FUND.PROTECAO DEFESA CONSUMIDOR-PROCON", "CONSUMER PROTECTION AND DEFENSE FOUNDATION - PROCON"),
    ("FUND.SISTEMA EST.ANALISE DADOS-SEADE", "STATE DATA ANALYSIS SYSTEM FOUNDATION - SEADE"),
    ("FUND.UNIV.VIRTUAL EST. S.P.-UNIVESP", "SP STATE VIRTUAL UNIVERSITY FOUNDATION - UNIVESP"),
    ("FUNDACAO DESENVOLV.ADMINISTRATIVO-FUNDAP", "ADMINISTRATIVE DEVELOPMENT FOUNDATION - FUNDAP"),
    ("FUNDACAO PARQUE ZOOLOGICO DE SAO PAULO", "SAO PAULO ZOOLOGICAL PARK FOUNDATION"),
    ("FUNDACAO PREFEITO FARIA LIMA-CEPAM", "MAYOR FARIA LIMA FOUNDATION - CEPAM"),
    ("GABINETE DO GOVERNADOR", "GOVERNOR'S OFFICE"),
    ("GESTAO PUBLICA", "PUBLIC MANAGEMENT"),
    ("HABITACAO", "HOUSING"),
    ("HOSPITAL CLINICAS FAC.MED.BOTUCATU-HCFMB", "BOTUCATU MEDICAL SCHOOL CLINICAL HOSPITAL - HCFMB"),
    ("HOSPITAL DAS CLINICAS FAC.MED. DA USP", "USP MEDICAL SCHOOL CLINICAL HOSPITAL"),
    ("HOSPITAL DAS CLINICAS FAC.MED. RIB.PRETO", "RIBEIRAO PRETO MEDICAL SCHOOL CLINICAL HOSPITAL"),
    ("IMPRENSA OFICIAL DO ESTADO S/A - IMESP", "STATE OFFICIAL PRESS - IMESP"),
    ("IMPRENSA OFICIAL DO ESTADO S/A-IMESP", "STATE OFFICIAL PRESS - IMESP"),
    ("INST MEDICINA SOC E CRIMIN S.P. -IMESC", "SP SOCIAL AND CRIMINAL MEDICINE INSTITUTE - IMESC"),
    ("INST.ASS.MED.SER.P.EST.-IAMSPE", "STATE PUBLIC SERVANT MEDICAL ASSISTANCE INSTITUTE - IAMSPE"),
    ("INST.ASSIST.MEDICA SERV.PUBLICO-IAMSPE", "STATE PUBLIC SERVANT MEDICAL ASSISTANCE INSTITUTE - IAMSPE"),
    ("INST.DE PAGTOS ESPECIAIS S. PAULO-IPESP", "SAO PAULO SPECIAL PAYMENTS INSTITUTE - IPESP"),
    ("INST.PESOS MEDIDAS EST.SAO PAULO-IPEM/SP", "SAO PAULO WEIGHTS AND MEASURES INSTITUTE - IPEM/SP"),
    ("INST.PESQUISAS TECN. EST.S.PAULO-IPT", "SAO PAULO STATE TECHNOLOGICAL RESEARCH INSTITUTE - IPT"),
    ("INSTITUTO DE ASSISTENCIA MEDICA SERVIDOR", "PUBLIC SERVANT MEDICAL ASSISTANCE INSTITUTE - IAMSPE"),
    ("JUNTA COMERC.E.S.PAULO-JUCESP", "SAO PAULO STATE BOARD OF TRADE - JUCESP"),
    ("JUNTA COMERCIAL ESTADO S.P.-JUCESP", "SAO PAULO STATE BOARD OF TRADE - JUCESP"),
    ("JUSTICA E DEFESA DA CIDADANIA", "JUSTICE AND CITIZENSHIP DEFENSE"),
    ("LOGISTICA E TRANSPORTES", "LOGISTICS AND TRANSPORTATION"),
    ("MEIO AMBIENTE", "ENVIRONMENT"),
    ("PLANEJAMENTO E DESENVOLVIMENTO REGIONAL", "REGIONAL PLANNING AND DEVELOPMENT"),
    ("POLICIA MILITAR ESTADO SAO PAULO", "SAO PAULO STATE MILITARY POLICE"),
    ("PROCURADORIA GERAL DO ESTADO", "STATE ATTORNEY GENERAL'S OFFICE"),
    ("SANEAMENTO E RECURSOS HIDRICOS", "SANITATION AND WATER RESOURCES"),
    ("SAO PAULO PREVIDENCIA-SPPREV", "SAO PAULO PENSION AUTHORITY - SPPREV"),
    ("SAUDE", "HEALTH"),
    ("SECRETARIA DE GOVERNO", "OFFICE OF GOVERNMENT"),
    ("SECRETARIA DE PLANEJAMENTO E GESTAO", "OFFICE OF PLANNING AND MANAGEMENT"),
    ("SEGURANCA PUBLICA", "PUBLIC SAFETY"),
    ("SUPERINT.TRAB.ARTES. COMUNIDADES-SUTACO", "ARTISAN AND COMMUNITY WORK SUPERINTENDENCY - SUTACO"),
    ("SUPERINTENDENCIA DE CONTROLE DE ENDEMIAS", "ENDEMIC DISEASE CONTROL SUPERINTENDENCY"),
    ("TRANSPORTES METROPOLITANOS", "METROPOLITAN TRANSPORTATION"),
    ("TURISMO", "TOURISM"),
], ["DEPARTMENT", "DEPARTMENT_EN"])

def clean_salary_df(df):
    # 1. Remove exact duplicates
    df = df.dropDuplicates()

    # 2. Drop rows missing critical fields
    df = df.dropna(subset=["NAME", "DEPARTMENT", "GROSS_TOTAL"])

    # 3. Fix BRL formatting: replace comma decimal → period, then cast to float
    df = df.withColumn(
        "GROSS_TOTAL",
        when(
            col("GROSS_TOTAL").rlike("^[0-9.,]+$"),
            regexp_replace(
                regexp_replace(col("GROSS_TOTAL"), "\\.", ""),
                ",",
                "."
            ).cast(FloatType())
        ).otherwise(None)
    )
    df = df.withColumn(
        "NET_TOTAL",
        when(
            col("NET_TOTAL").rlike("^[0-9.,]+$"),
            regexp_replace(
                regexp_replace(col("NET_TOTAL"), "\\.", ""),
                ",",
                "."
            ).cast(FloatType())
        ).otherwise(None)
    )

    # 4. Standardize text fields
    df = df.withColumn("NAME", trim(upper(col("NAME"))))
    df = df.withColumn("DEPARTMENT", trim(upper(col("DEPARTMENT"))))
    df = df.withColumn("POSITION", trim(upper(col("POSITION"))))

    # 5. Generate surrogate keys (sha2 hash)
    df = df.withColumn("EMPLOYEE_ID",
            sha2(concat_ws("|", col("NAME"), col("POSITION"), col("DEPARTMENT")), 256))
    df = df.withColumn("DEPARTMENT_ID",
            sha2(col("DEPARTMENT"), 256))

    # 6. Rename to match Gold-layer / AI-module naming convention
    df = df.withColumnRenamed("GROSS_TOTAL", "GROSS_SALARY")
    df = df.withColumnRenamed("NET_TOTAL", "NET_SALARY")

    # 7. NEW: Join English department translation
    df = df.join(department_translation, on="DEPARTMENT", how="left")

    return df

df_silver_2013 = clean_salary_df(df_2013)
df_silver_2014 = clean_salary_df(df_2014)
df_silver_2015 = clean_salary_df(df_2015)

# Combine all years
df_silver = df_silver_2013.unionByName(df_silver_2014).unionByName(df_silver_2015)

df_silver.write.mode('overwrite').parquet(f'{base}/data/silver/all_years')
print(f"Silver rows: {df_silver.count()}")

Silver rows: 37409968


In [ ]:
# df_silver.printSchema()

root
 |-- DEPARTMENT: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- POSITION: string (nullable = true)
 |-- MONTH_TOTAL: string (nullable = true)
 |-- GROSS_SALARY: float (nullable = true)
 |-- NET_SALARY: float (nullable = true)
 |-- MONTH: string (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- EMPLOYEE_ID: string (nullable = true)
 |-- DEPARTMENT_ID: string (nullable = true)
 |-- DEPARTMENT_EN: string (nullable = true)



# 5.0 Gold Layer: Integrate & Aggregate

In [ ]:
from pyspark.sql.functions import avg, sum as spark_sum, max as spark_max
from pyspark.sql import Window
from pyspark.sql.functions import lag

# Read silver
df_silver = spark.read.parquet(f'{base}/data/silver/all_years')

# Fact table (salary transactions)
fact_salaries = df_silver.select(
    "EMPLOYEE_ID", "DEPARTMENT_ID", "YEAR", "MONTH",
    "GROSS_SALARY", "NET_SALARY"
)

# Dimension: Employees
dim_employees = df_silver.select("EMPLOYEE_ID", "NAME", "POSITION").dropDuplicates(["EMPLOYEE_ID"])

# Dimension: Departments (includes English translation)
dim_departments = df_silver.select("DEPARTMENT_ID", "DEPARTMENT", "DEPARTMENT_EN").dropDuplicates(["DEPARTMENT_ID"])

# Aggregation: Dept avg salary per year
dept_avg = df_silver.groupBy("DEPARTMENT_ID", "YEAR") \
    .agg(avg("GROSS_SALARY").alias("AVG_GROSS"), spark_sum("GROSS_SALARY").alias("TOTAL_BUDGET"))

# Write Gold tables
fact_salaries.write.mode('overwrite').partitionBy("YEAR").parquet(f'{base}/data/gold/fact_salaries')
dim_employees.write.mode('overwrite').parquet(f'{base}/data/gold/dim_employees')
dim_departments.write.mode('overwrite').parquet(f'{base}/data/gold/dim_departments')
dept_avg.write.mode('overwrite').parquet(f'{base}/data/gold/dept_avg')

# 6.0 AI Module: Isolation Forest

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql.functions import monotonically_increasing_id
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

# Read Gold fact table
df_gold = spark.read.parquet(f'{base}/data/gold/fact_salaries') \
    .join(dim_employees, "EMPLOYEE_ID") \
    .join(dim_departments, "DEPARTMENT_ID")

# Encode categoricals
indexer_pos = StringIndexer(inputCol="POSITION", outputCol="POSITION_IDX", handleInvalid="keep")
indexer_dep = StringIndexer(inputCol="DEPARTMENT", outputCol="DEPARTMENT_IDX", handleInvalid="keep")

df_indexed = indexer_pos.fit(df_gold).transform(df_gold)
df_indexed = indexer_dep.fit(df_indexed).transform(df_indexed)

# Add a stable unique row ID
df_indexed = df_indexed.withColumn("ROW_ID", monotonically_increasing_id())

# Sample before pulling to driver
features = ["GROSS_SALARY", "POSITION_IDX", "DEPARTMENT_IDX"]
df_pd = df_indexed.select(["ROW_ID"] + features).dropna().sample(fraction=0.1, seed=42).toPandas()

print(f"Sampled rows for Isolation Forest: {len(df_pd)}")

# Train Isolation Forest
clf = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
df_pd["IS_ANOMALY"] = clf.fit_predict(df_pd[features])
df_pd["IS_ANOMALY"] = df_pd["IS_ANOMALY"].map({1: False, -1: True})

# Build Spark DataFrame of anomaly results
df_anomaly = spark.createDataFrame(df_pd[["ROW_ID", "IS_ANOMALY"]])

# INNER join — only keep the rows that were actually scored (3.7M, not 37M)
df_final = df_indexed.join(df_anomaly, "ROW_ID", "inner").drop("ROW_ID")

# Write final Gold with anomaly flag — now only writing the scored subset
df_final.write.mode('overwrite').partitionBy("YEAR") \
    .parquet(f'{base}/data/gold/fact_salaries_anomaly')

print("Anomaly detection complete.")
print(f"Flagged records: {df_pd['IS_ANOMALY'].sum()}")
print(f"Total records written: {df_pd.shape[0]}")

Sampled rows for Isolation Forest: 3742269
Anomaly detection complete.
Flagged records: 187114
Total records written: 3742269


# 7.0 Export for Power BI

In [ ]:
# Export aggregated tables as CSV for Power BI import — write directly via Spark, no toPandas()
df_final.coalesce(1).write.mode('overwrite').option("header", True).csv(f'{base}/data/output/fact_salaries_anomaly_csv')
dept_avg.coalesce(1).write.mode('overwrite').option("header", True).csv(f'{base}/data/output/dept_avg_csv')
dim_employees.coalesce(1).write.mode('overwrite').option("header", True).csv(f'{base}/data/output/dim_employees_csv')
dim_departments.coalesce(1).write.mode('overwrite').option("header", True).csv(f'{base}/data/output/dim_departments_csv')

print("CSV outputs ready for Power BI.")

CSV outputs ready for Power BI.


In [ ]:
import glob, shutil

def flatten_csv_output(folder_path, final_name):
    part_file = glob.glob(f"{folder_path}/part-*.csv")[0]
    shutil.move(part_file, f"{base}/data/output/{final_name}")
    shutil.rmtree(folder_path)

flatten_csv_output(f'{base}/data/output/fact_salaries_anomaly_csv', 'fact_salaries_anomaly.csv')
flatten_csv_output(f'{base}/data/output/dept_avg_csv', 'dept_avg.csv')
flatten_csv_output(f'{base}/data/output/dim_employees_csv', 'dim_employees.csv')
flatten_csv_output(f'{base}/data/output/dim_departments_csv', 'dim_departments.csv')

print("Flattened CSVs ready for Power BI import.")

Flattened CSVs ready for Power BI import.


In [ ]:
import os
for f in ['fact_salaries_anomaly.csv', 'dept_avg.csv', 'dim_employees.csv', 'dim_departments.csv']:
    path = f'{base}/data/output/{f}'
    print(f, os.path.exists(path), os.path.getsize(path) if os.path.exists(path) else 'MISSING')

fact_salaries_anomaly.csv True 1015133279
dept_avg.csv True 30428
dim_employees.csv True 190933826
dim_departments.csv True 15062
